# Лабораторная 3 Визуализация и EDA

**Иноземцев Сергей Викторович · ЕТ-216 · вариант 11**

Источник задания: «Лб 3.docx» и StudentsPerformance.csv. Все вычисления воспроизводимы при последовательном запуске ячеек. Фиксированные random seed заданы в коде.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'datasets'/'StudentsPerformance.csv').is_file())
DATA = BASE/'datasets'
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
def show(name):
    plt.tight_layout()
    plt.savefig(FIG/(name+'.png'), bbox_inches='tight')
    plt.show()

## Проверка исходных данных StudentsPerformance
Реальный файл из вложения курса. В условии опечатка gender=mail; используется существующее значение male.

In [ ]:
df=pd.read_csv(DATA/'StudentsPerformance.csv');scores=['math score','reading score','writing score']
print(df.shape);df.info();display(df.head());display(df.describe(include='all'));display(df.isna().sum());print('Полных совпадений строк:',df.duplicated().sum())
assert df[scores].ge(0).all().all() and df[scores].le(100).all().all()

## Pandas Три диаграммы

In [ ]:
df['parental level of education'].value_counts().plot.bar();plt.title('Образование родителей');plt.xlabel('Уровень');plt.ylabel('Число учеников');show('parent_education')
df['race/ethnicity'].value_counts().plot.pie(autopct='%1.1f%%');plt.title('Группы race/ethnicity');plt.ylabel('');show('ethnicity')
df[scores].hist(bins=15,figsize=(13,4));plt.suptitle('Распределения оценок');show('score_histograms')

## Matplotlib Сетка 3 на 2

In [ ]:
fig=plt.figure(figsize=(16,17))
ax=fig.add_subplot(321);x=np.linspace(-2*np.pi,2*np.pi,500);ax.plot(x,np.cos(2*x),color='yellow', linewidth=2,label='cos(2x)');ax.plot(x,2*np.cos(x),'r',label='2cos(x)');ax.grid();ax.legend();ax.set(title='Тригонометрические функции',xlabel='x',ylabel='y')
ax=fig.add_subplot(322);ax.boxplot(df['math score']);ax.set(title='Математика',ylabel='Баллы')
ax=fig.add_subplot(323);cnt=df['race/ethnicity'].value_counts().sort_index();ax.bar(cnt.index,cnt.values,alpha=.5);ax.plot(cnt.index,cnt.values,'o-',color='red');ax.set(title='Столбцы и линия',xlabel='Группа',ylabel='Учеников')
ax=fig.add_subplot(324);ax.text(.5,.5,'HELLO!!!',ha='center',va='center',fontsize=35);ax.axis('off')
ax=fig.add_subplot(325,projection='3d');ax.scatter(df[scores[0]],df[scores[1]],df[scores[2]],s=10,alpha=.3);ax.set(xlabel='Математика',ylabel='Чтение',zlabel='Письмо',title='Три оценки')
ax=fig.add_subplot(326);df.loc[df.gender.eq('male'),'parental level of education'].value_counts().plot.pie(ax=ax,autopct='%1.1f%%');ax.set(title='Образование родителей юношей',ylabel='')
show('matplotlib_grid')

## Seaborn Парные графики и корреляции

In [ ]:
sns.pairplot(df[scores],diag_kind='hist');plt.suptitle('Оценки и парные зависимости',y=1.02);show('pair_hist')
sns.pairplot(df[scores+['gender']],hue='gender',diag_kind='kde');plt.suptitle('Оценки с группировкой по полу',y=1.02);show('pair_gender')
sns.heatmap(df[scores].corr(),annot=True,vmin=-1,vmax=1,cmap='coolwarm');plt.title('Корреляция оценок');show('heatmap')

## Аномалии и вывод

In [ ]:
q1,q3=df['math score'].quantile([.25,.75]);iqr=q3-q1
outliers=df[(df['math score']<q1-1.5*iqr)|(df['math score']>q3+1.5*iqr)]
print('Границы IQR:',q1-1.5*iqr,q3+1.5*iqr,'Выбросов:',len(outliers));display(outliers)
display(df.groupby('test preparation course')[scores].mean());display(df.groupby('gender')[scores].mean())

В файле 1000 учеников, 0 пропусков, 0 полных совпадений строк. Оценки укладываются в 0–100. По правилу 1,5 IQR выявлено 8 экстремальных оценок математики; это допустимые низкие баллы, а не автоматически ошибки. Самая сильная корреляция — между чтением и письмом: r=0.955. Наблюдаемые различия групп не доказывают причинного влияния пола, этнической группы или подготовки. Для этого нужны контроль смешивающих факторов и отдельный дизайн исследования.